In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
from sklearn.model_selection import train_test_split, KFold,cross_validate, StratifiedKFold,GridSearchCV,  RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree 
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from scipy.stats import randint
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline

df = pd.read_csv('train (1).csv')
df


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,1456,60,RL,62.0,7917,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2007,WD,Normal,175000
1456,1457,20,RL,85.0,13175,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,2,2010,WD,Normal,210000
1457,1458,70,RL,66.0,9042,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,GdPrv,Shed,2500,5,2010,WD,Normal,266500
1458,1459,20,RL,68.0,9717,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,142125


In [2]:
#버려, 결측치가 1400개 정도 됨, 의미 없음
df = df.drop(['PoolQC', 'MiscFeature', 'Alley'], axis=1)

In [3]:
df

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,...,3SsnPorch,ScreenPorch,PoolArea,Fence,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,Reg,Lvl,AllPub,Inside,...,0,0,0,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,Reg,Lvl,AllPub,FR2,...,0,0,0,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,IR1,Lvl,AllPub,Inside,...,0,0,0,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,IR1,Lvl,AllPub,Corner,...,0,0,0,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,IR1,Lvl,AllPub,FR2,...,0,0,0,NaN,0,12,2008,WD,Normal,250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,1456,60,RL,62.0,7917,Pave,Reg,Lvl,AllPub,Inside,...,0,0,0,NaN,0,8,2007,WD,Normal,175000
1456,1457,20,RL,85.0,13175,Pave,Reg,Lvl,AllPub,Inside,...,0,0,0,MnPrv,0,2,2010,WD,Normal,210000
1457,1458,70,RL,66.0,9042,Pave,Reg,Lvl,AllPub,Inside,...,0,0,0,GdPrv,2500,5,2010,WD,Normal,266500
1458,1459,20,RL,68.0,9717,Pave,Reg,Lvl,AllPub,Inside,...,0,0,0,NaN,0,4,2010,WD,Normal,142125


In [4]:
#진짜 집에 없는 것들, None으로 처리
none_cols = [
    'Fence','FireplaceQu','GarageType','GarageFinish','GarageQual','GarageCond',
    'BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2', 'MasVnrType'
]

for col in none_cols:
    df[col] = df[col].fillna('None')

#데이터 자체가 비어있는 것 , 중앙값으로 처리 
num_cols = ['LotFrontage','MasVnrArea','GarageYrBlt']

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

#하나 비어있는것, 최빈값으로 처리 
df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])

In [5]:
# 원-핫 인코딩 (우선은 특성공학X)

# 등급별 매핑
qual_map = {
    'None': 0,
    'Po': 1,
    'Fa': 2,
    'TA': 3,
    'Gd': 4,
    'Ex': 5
}

qual_cols = [
    'ExterQual','ExterCond','BsmtQual','BsmtCond',
    'HeatingQC','KitchenQual','FireplaceQu',
    'GarageQual','GarageCond'
]

df[qual_cols] = df[qual_cols].fillna('None')
df['BsmtExposure'] = df['BsmtExposure'].fillna('None')
df['GarageFinish'] = df['GarageFinish'].fillna('None')

for col in qual_cols:
    df[col] = df[col].map(qual_map)



# 기타 매핑
exposure_map = {
    'None': 0,
    'No': 1,
    'Mn': 2,
    'Av': 3,
    'Gd': 4
}

df['BsmtExposure'] = df['BsmtExposure'].map(exposure_map)

garage_map = {
    'None': 0,
    'Unf': 1,
    'RFn': 2,
    'Fin': 3
}

df['GarageFinish'] = df['GarageFinish'].map(garage_map)
df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
df['Qual_x_GrLivArea'] = df['OverallQual'] * df['GrLivArea']
df['HouseAge'] = df['YrSold'] - df['YearBuilt']
#원핫인코딩
X = df.drop('SalePrice', axis=1)
y = df['SalePrice']

cat_cols = X.select_dtypes(include='object').columns
X = pd.get_dummies(X, columns=cat_cols, drop_first=False) #우선 false로 

print(X.shape)
print(X.info()) 

(1460, 250)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Columns: 250 entries, Id to SaleCondition_Partial
dtypes: bool(199), float64(3), int64(48)
memory usage: 865.6 KB
None


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print(X_train.columns)

Index(['Id', 'MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual',
       'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'ExterQual',
       ...
       'SaleType_ConLw', 'SaleType_New', 'SaleType_Oth', 'SaleType_WD',
       'SaleCondition_Abnorml', 'SaleCondition_AdjLand',
       'SaleCondition_Alloca', 'SaleCondition_Family', 'SaleCondition_Normal',
       'SaleCondition_Partial'],
      dtype='object', length=250)


In [7]:
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import Lasso, Ridge

RFmodel = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

HGMmodel = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

XGBmodel = XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

LGBMmodel = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=-1
)

LSmodel = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Lasso(alpha=0.001, max_iter=30000))
])

Rmodel = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0))
])

lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

In [8]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# -----------------------------------
# 1. 모델 모음
# -----------------------------------
models = {
    "RandomForest": RFmodel,
    "HistGB": HGMmodel,
    "XGBoost": XGBmodel,
    "LightGBM": LGBMmodel,
    "Lasso": LSmodel,
    "Ridge": Rmodel,
    "LinearRegression" :  lr
}

# -----------------------------------
# 2. 결과 저장용 리스트
# -----------------------------------
results_list = []

# -----------------------------------
# 3. 각 모델별 학습 / 예측 / 평가
# -----------------------------------
for name, model in models.items():
    temp_model = clone(model)
    
    # 로그 변환해서 학습
    temp_model.fit(X_train, np.log1p(y_train))
    
    # 예측
    y_pred_log = temp_model.predict(X_test)
    
    # 원래 값으로 복원
    y_pred = np.expm1(y_pred_log)
    y_pred = np.clip(y_pred, 0, None)
    
    # 평가 지표 계산
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    rmsle = np.sqrt(mean_squared_error(np.log1p(y_test), np.log1p(y_pred)))
    r2 = r2_score(y_test, y_pred)
    
    # 결과 저장
    results_list.append({
        "Model": name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "RMSLE": rmsle,
        "R2": r2
    })

# -----------------------------------
# 4. DataFrame 생성
# -----------------------------------
results_df = pd.DataFrame(results_list)

# RMSE 기준 정렬 (낮을수록 좋음)
results_df = results_df.sort_values(by="RMSE", ascending=True)

# 보기 좋게 반올림
results_df = results_df.round(4)

# 인덱스 정리
results_df = results_df.reset_index(drop=True)

results_df

,Model,MAE,MSE,RMSE,RMSLE,R2
0,Lasso,14322.5549,4.812072e+08,21936.4362,0.1205,0.9373
1,Ridge,15186.6357,5.178828e+08,22757.0376,0.1234,0.9325
2,LinearRegression,15264.9272,5.250577e+08,22914.1384,0.1238,0.9315
3,XGBoost,15401.6611,7.160918e+08,26759.8912,0.1358,0.9066
4,LightGBM,15676.9427,8.164231e+08,28573.1191,0.1365,0.8936
5,RandomForest,17102.5896,8.750816e+08,29581.7784,0.1432,0.8859
6,HistGB,16876.4941,9.188044e+08,30311.7865,0.1425,0.8802
